# 11.1 · 文本预处理 / Text Preprocessing

> **课程定位 / Where this fits**
> 第 1 课，**Part 11 · 经典 NLP**。这是所有文本任务的第一步。
> Lesson 1, **Part 11 · Classic NLP**. The first step of every text task.
>
> 模型不能直接"读"原始文字——它需要**干净、结构化的词元(token)**。原始文本充满大小写、标点、数字、停用词、词形变化等噪声。**文本预处理**就是把杂乱的字符串变成规整的 token 序列：**分词 → 归一化 → 去停用词 → 词干/词形还原**。这一步看似琐碎，却直接决定下游(向量化、分类)的质量——"garbage in, garbage out"。本课用真实语料 + **大量可视化**(含 Zipf 定律)讲透每一步。
> Models can't "read" raw text — they need **clean, structured tokens**. Raw text is full of noise: case, punctuation, numbers, stopwords, word-form variation. **Text preprocessing** turns messy strings into tidy token sequences: **tokenize → normalize → remove stopwords → stem/lemmatize**. Seemingly mundane, it directly governs downstream quality — "garbage in, garbage out." We teach each step with real corpora + **rich visualization** (incl. Zipf's law).
>
> 💼 **实战/面试视角**："分词的难点 / 词干 vs 词形还原 / 要不要去停用词 / Zipf 定律" 是 NLP 入门必问。
> 💼 **Practical/interview angle:** "tokenization challenges / stemming vs lemmatization / remove stopwords or not / Zipf's law" — NLP entry must-knows.

> 📐 **符号约定 / Notation**
> - token —— 文本切出的最小单位(词/子词) / a token: smallest unit (word/subword)
> - 词表(vocabulary) —— 语料中所有不同 token 的集合 / set of unique tokens
> - 词干(stem)/词形(lemma) —— 词的粗略词根 / 词典里的原型 / crude root / dictionary base form

> 💡 **面试相关 / Interview-relevant**
> - "词干提取 vs 词形还原的区别"（出镜率 ★★★★★）
> - "为什么/何时去停用词"（★★★★）
> - "分词在中文/英文的不同难点"（★★★）
> - "Zipf 定律是什么、有何影响"（★★★）

---

## 学习目标 / Learning Objectives
1. 理解预处理的目的与"垃圾进垃圾出"。
   Understand the goal of preprocessing and "garbage in, garbage out."
2. 掌握**分词**(正则 vs 工具)与归一化。
   Master tokenization (regex vs tools) and normalization.
3. 会用**停用词**并理解其利弊。
   Use stopwords and understand trade-offs.
4. 分清**词干提取 vs 词形还原**。
   Distinguish stemming vs lemmatization.
5. 观察 **Zipf 定律** 与各步对词表的影响。
   Observe Zipf's law and each step's effect on vocabulary.

## 目录 / TOC
1. [为什么要预处理：看看原始文本 ⭐](#1)
2. [分词 Tokenization ⭐](#2)
3. [归一化与停用词 ⭐](#3)
4. [词干提取 vs 词形还原 ⭐](#4)
5. [完整流程 + Zipf 定律 + 小结 ⭐](#5)


<a id="1"></a>
## 1. 为什么要预处理：看看原始文本 ⭐ / Why Preprocess: Look at Raw Text

我们用经典的 **20 Newsgroups**(2 万篇新闻组帖子)语料。先看一篇**原始文本**——你会看到大小写混杂、标点、邮件头、数字、换行等一堆"噪声"。模型若直接吃这些，会把 `"Dog"`、`"dog"`、`"dog,"`、`"dogs"` 当成 4 个不同的东西，浪费且稀疏。
We use the classic **20 Newsgroups** corpus (~20k newsgroup posts). First, a **raw document** — mixed case, punctuation, headers, numbers, newlines, lots of "noise." Fed directly, a model treats `"Dog"`, `"dog"`, `"dog,"`, `"dogs"` as 4 different things — wasteful and sparse.


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns, re
from collections import Counter
from sklearn.datasets import fetch_20newsgroups
sns.set_theme(style="whitegrid")

# 取两个类别的帖子作演示 / fetch posts from two categories
data = fetch_20newsgroups(subset="train", categories=["sci.space", "rec.sport.baseball"],
                          remove=("headers", "footers", "quotes"))   # 去掉邮件头/尾/引用 / strip headers etc.
docs = data.data
print(f"语料: {len(docs)} 篇文档")
print("="*70)
print("一篇原始文本(节选):")
print(docs[2][:400])
print("="*70)
print("注意: 大小写混杂、标点、数字、换行 → 模型会把 Dog/dog/dog,/dogs 当成不同 token")


<a id="2"></a>
## 2. 分词 Tokenization ⭐ / Tokenization

**分词(tokenization)**：把连续的字符串切成一个个 **token**(通常是词)。这是 NLP 的第一步，看似简单实则有坑：
**Tokenization:** split a continuous string into **tokens** (usually words). The first NLP step — deceptively tricky:
- 标点怎么处理？`"don't"` 切成 `don't` 还是 `do` + `n't`？`"U.S.A."` 呢？
  Punctuation? Is `"don't"` one token or `do` + `n't`? What about `"U.S.A."`?
- 英文靠空格分词相对容易；**中文没有空格**，需要专门的分词算法(如 jieba)。
  English splits on spaces; **Chinese has no spaces**, needing dedicated segmenters (e.g. jieba).
- 现代大模型用**子词(subword)** 分词(BPE/WordPiece)，把罕见词拆成子词片段(Part 12 详讲)。
  Modern LLMs use **subword** tokenization (BPE/WordPiece), splitting rare words into pieces (detailed in Part 12).

下面对比**用正则从零分词** 和 **nltk 的分词器**。
Below we compare **regex tokenization from scratch** vs **nltk's tokenizer**.


In [ ]:
import nltk
for r in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(r, quiet=True)                          # 下载所需数据(很小, 已缓存则跳过) / fetch nltk data
from nltk.tokenize import word_tokenize

sample = "The U.S.A.'s rockets don't fail often! Visit NASA in 2024 :)"
# 1) 从零: 用正则提取"单词字符序列" / from scratch: regex grab word-character runs
regex_tokens = re.findall(r"[a-zA-Z]+", sample)          # 只保留字母串(最简单的分词) / letter runs only
# 2) nltk: 更聪明(处理缩写/标点) / nltk: smarter (handles contractions/punctuation)
nltk_tokens = word_tokenize(sample)
print("原句:", sample)
print("\n正则分词(只取字母串):", regex_tokens)
print("nltk 分词(更细致):    ", nltk_tokens)
print("\n区别: 正则简单粗暴(丢标点/缩写); nltk 把 don't→do n't, 保留标点 → 更接近语言学单位")
print("实战: 英文常用 nltk/spaCy; 大模型用子词(BPE); 中文用 jieba 等")


<a id="3"></a>
## 3. 归一化与停用词 ⭐ / Normalization & Stopwords

**归一化(normalization)**：统一形式，减少无意义的差异。
**Normalization:** unify forms to remove meaningless variation.
- **小写化**：`"Dog"`→`"dog"`(否则句首的词和句中同一个词被当成两个)。
  **Lowercasing:** `"Dog"`→`"dog"` (else sentence-start vs mid-sentence are two tokens).
- **去标点/数字**：很多任务里它们是噪声(但情感分析里"!"可能有用——要看任务)。
  **Strip punctuation/numbers:** noise for many tasks (but "!" may matter for sentiment — task-dependent).

**停用词(stopwords)**：`the, is, a, of` 这类**超高频但信息量低**的词。去掉它们能**缩小词表、突出关键词**。
**Stopwords:** ultra-frequent, low-information words like `the, is, a, of`. Removing them **shrinks vocabulary and highlights keywords**.

> ⚠️ **要不要去停用词(面试)**：**看任务**。词袋/主题模型/检索通常去；但**情感分析**("not good"里的 not)、**机器翻译**、**现代 Transformer**通常**不去**——停用词承载语法和否定信息。
> ⚠️ **Remove stopwords or not (interview):** **depends on the task.** BoW/topic-modeling/retrieval usually do; but **sentiment** ("not good"), **MT**, and **modern Transformers** usually **don't** — stopwords carry grammar and negation.


In [ ]:
from nltk.corpus import stopwords
stop = set(stopwords.words("english"))                   # 英文停用词表(~180个) / English stopword list
print(f"停用词表大小: {len(stop)}; 例: {sorted(list(stop))[:12]}")

def normalize(text):
    text = text.lower()                                  # 小写 / lowercase
    tokens = re.findall(r"[a-z]+", text)                 # 取字母串(去标点/数字) / letters only
    tokens = [t for t in tokens if len(t) > 2]           # 去掉过短词 / drop very short tokens
    return tokens

raw_tokens = normalize(docs[2])
no_stop = [t for t in raw_tokens if t not in stop]       # 去停用词 / remove stopwords
print(f"\n归一化后 token 数: {len(raw_tokens)}")
print(f"再去停用词后:     {len(no_stop)}  (减少 {(1-len(no_stop)/len(raw_tokens))*100:.0f}%)")
print(f"\n去停用词前(前15): {raw_tokens[:15]}")
print(f"去停用词后(前15): {no_stop[:15]}  ← the/is/of 等被去掉, 剩下更有信息量的词")


<a id="4"></a>
## 4. 词干提取 vs 词形还原 ⭐ / Stemming vs Lemmatization

同一个词有很多形态：`run, runs, running, ran`。我们常想把它们**归并成一个**，减少词表、让"含义相同"的词被同等对待。两种方法(面试超高频对比)：
A word has many forms: `run, runs, running, ran`. We often want to **collapse them into one** to shrink vocabulary and treat same-meaning words alike. Two methods (very high-frequency interview comparison):

- **词干提取(stemming)**：用**粗暴的规则砍词缀**得到"词干"。快，但词干**可能不是真词**：`studies→studi`、`running→run`、`better→better`(砍不动)。代表：Porter Stemmer。
  **Stemming:** crudely **chop affixes** by rules to get a "stem." Fast, but the stem **may not be a real word**: `studies→studi`, `running→run`, `better→better` (can't fix). E.g. Porter Stemmer.
- **词形还原(lemmatization)**：用**词典 + 词性**找出**真正的原型(lemma)**：`studies→study`、`ran→run`(需告知是动词)、`better→good`。更准但更慢。
  **Lemmatization:** use a **dictionary + part-of-speech** to find the **true base form (lemma)**: `studies→study`, `ran→run` (needs "verb"), `better→good`. More accurate, slower.

**一句话记忆**：词干提取=砍词缀(快/糙/可能非真词)；词形还原=查词典找原型(慢/准/真词)。
**One-liner:** stemming = chop affixes (fast/crude/maybe non-word); lemmatization = dictionary base form (slow/accurate/real word).


In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer
stemmer = PorterStemmer(); lemm = WordNetLemmatizer()
words = ["running", "runs", "ran", "studies", "studying", "better", "feet", "cats", "organization"]
print(f"{'原词 word':<16}{'词干 stem':<14}{'词形还原 lemma':<14}")
print("-"*44)
for w in words:
    s = stemmer.stem(w)
    l = lemm.lemmatize(w, pos="v")                       # 按动词还原(pos 很重要) / lemmatize as verb (pos matters!)
    print(f"{w:<16}{s:<14}{l:<14}")
print("\n观察: stemming 砍出的词干可能不是真词(studi); lemma 是真正原型(study, run)")
print("注意: 词形还原需要词性(pos)才准, 如 ran→run 必须知道它是动词")


<a id="5"></a>
## 5. 完整流程 + Zipf 定律 + 小结 ⭐ / Full Pipeline + Zipf's Law

把各步串成一个**预处理流程**，并可视化两个重要现象：
Chain the steps into a **preprocessing pipeline** and visualize two important phenomena:
1. **各步如何缩小词表**：分词 → 去停用词 → 词形还原，词表逐步变小、变干净。
   **How each step shrinks vocabulary:** tokenize → remove stopwords → lemmatize, progressively smaller/cleaner.
2. **Zipf 定律(面试常考)**：自然语言里，**第 $r$ 高频的词，其频率约正比于 $1/r$**。即极少数词(the, of...)占据绝大部分出现次数，绝大多数词都很罕见(长尾)。在 log-log 图上近似一条直线。这解释了为什么去停用词有用、为什么文本向量极度稀疏。
   **Zipf's law:** in natural language, **the $r$-th most frequent word has frequency ∝ $1/r$**. A few words (the, of…) dominate occurrences; most words are rare (long tail). On a log-log plot it's nearly a straight line. This explains why stopword removal helps and why text vectors are so sparse.


In [ ]:
lemm2 = WordNetLemmatizer()
def preprocess(text):
    tokens = re.findall(r"[a-z]+", text.lower())         # 小写+取字母串 / lowercase + letters
    tokens = [t for t in tokens if len(t) > 2 and t not in stop]   # 去短词+停用词 / drop short + stopwords
    return [lemm2.lemmatize(t) for t in tokens]          # 词形还原 / lemmatize

# 各阶段词表大小 / vocabulary size after each stage
all_text = " ".join(docs)
v_tokens = set(re.findall(r"[a-z]+", all_text.lower()))                       # 仅分词归一化 / tokenize only
v_nostop = set(t for t in v_tokens if len(t)>2 and t not in stop)             # +去停用词 / +stopwords
v_lemma  = set(lemm2.lemmatize(t) for t in v_nostop)                          # +词形还原 / +lemmatize

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
# 左: 词表缩减 / vocab reduction
axes[0].bar(["分词归一化", "+去停用词", "+词形还原"], [len(v_tokens), len(v_nostop), len(v_lemma)],
            color=["#bbb","#5a9","#39c"])
for i,v in enumerate([len(v_tokens),len(v_nostop),len(v_lemma)]): axes[0].text(i, v+200, f"{v:,}", ha="center")
axes[0].set_ylabel("词表大小(不同词数)"); axes[0].set_title("各预处理步骤逐步缩小词表")
# 右: Zipf 定律 / Zipf's law
freqs = Counter(re.findall(r"[a-z]+", all_text.lower()))
ranked = sorted(freqs.values(), reverse=True)
axes[1].loglog(range(1, len(ranked)+1), ranked, lw=2)
axes[1].set_xlabel("词频排名 rank (对数)"); axes[1].set_ylabel("出现次数 (对数)")
axes[1].set_title("Zipf 定律: 频率 ∝ 1/排名 (log-log 近似直线)")
plt.tight_layout(); plt.show()
top = freqs.most_common(8)
print("最高频的8个词(大多是停用词):", [w for w,_ in top])
print(f"完整预处理示例: {preprocess(docs[2])[:12]}")
print("Zipf: 极少数词占绝大多数出现 → 去停用词去掉高频低信息词; 长尾导致文本向量极稀疏")


```
预处理目的: 原始文本→干净token序列; garbage in garbage out
分词: 切成token; 英文靠空格(nltk/spaCy), 中文需分词(jieba), 大模型用子词(BPE)
归一化: 小写/去标点数字/去短词; 减少无意义差异
停用词: the/is/of 高频低信息; 去否看任务(BoW/主题/检索去; 情感/翻译/Transformer不去)
词干 stemming: 砍词缀, 快/糙/可能非真词(studi); 词形还原 lemma: 查词典找原型, 慢/准/真词(study), 需pos
Zipf定律: 频率∝1/排名; 少数词占多数出现, 长尾→文本向量极稀疏
```

### 💡 面试速查 / Interview cheat-sheet
1. **词干 vs 词形还原**: 砍词缀(快糙非真词) vs 查词典原型(慢准真词,需pos)。
   Stemming vs lemmatization: chop affixes (fast/crude) vs dictionary base form (accurate, needs pos).
2. **停用词**: 高频低信息; 去否看任务(情感/Transformer 通常不去)。
   Stopwords: high-freq/low-info; removal is task-dependent (keep for sentiment/Transformers).
3. **分词难点**: 标点/缩写/无空格语言(中文)/子词。
   Tokenization challenges: punctuation/contractions/no-space languages/subwords.
4. **Zipf 定律**: 频率∝1/排名; 少数词主导, 长尾稀疏。
   Zipf: frequency ∝ 1/rank; few words dominate, long-tail sparsity.
5. **顺序**: 分词→归一化→去停用词→词干/词形还原。
   Order: tokenize → normalize → stopwords → stem/lemmatize.

### 下一节 / Next
**11.2 词袋与 TF-IDF**——清洗后的 token 怎么变成模型能算的**数字向量**？最经典的两种方法：词袋(数词频)和 TF-IDF(给"既高频又独特"的词更高权重)。我们会**从零实现 TF-IDF**。
**11.2 BoW & TF-IDF** — how do clean tokens become **numeric vectors**? The two classic methods: Bag-of-Words (count words) and TF-IDF (upweight words that are frequent *and* distinctive). We'll **implement TF-IDF from scratch**.
